# Phase 3 item 1 (data prep) — vector search validation

Tests `vector_db.chunks_docs_embedded` against real `VECTOR_SEARCH` queries
(`docs/data-pipeline.md` Part 3, "What `search_docs` runs"), then visualizes
the embedding space to sanity-check that retrieval is actually finding
semantically related chunks — not just running without error.

In [1]:
from google.cloud import bigquery

PROJECT = "instacart-ml-model"
client = bigquery.Client(project=PROJECT)

## `vector_search()` — same shape as the documented `search_docs` query

In [2]:
SEARCH_SQL = f"""
SELECT base.chunk_text, base.file_path, base.section, distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT}.vector_db.chunks_docs_embedded`, 'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `{PROJECT}.staging.embedding_model`,
     (SELECT @query AS content),
     STRUCT(TRUE AS flatten_json_output))),
  top_k => @top_k, distance_type => 'COSINE')
ORDER BY distance
"""


def vector_search(query_text: str, top_k: int = 5):
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("query", "STRING", query_text),
        bigquery.ScalarQueryParameter("top_k", "INT64", top_k),
    ])
    return list(client.query(SEARCH_SQL, job_config=job_config).result())

## Real questions, eyeball whether the top match actually makes sense

In [4]:
TEST_QUERIES = [
    "How do I set up GCP permissions for this project?",
    "What metrics were used to evaluate the model?",
    "How is financial impact calculated from the model's predictions?",
    "What machine learning models were used and how were they combined?",
    "What happens when a new user has no order history?",
]

for q in TEST_QUERIES:
    print("=" * 80)
    print("QUERY:", q)
    for row in vector_search(q, top_k=5):
        print(f"  [{row.distance:.4f}] {row.section}")
        print(f"           {row.chunk_text[:150].replace(chr(10), ' ')}...")

QUERY: How do I set up GCP permissions for this project?
  [0.2606] Environment Setup & Provisioning > 2. IAM Security & Permissions Configuration
           [Environment Setup & Provisioning > 2. IAM Security & Permissions Configuration] 2. IAM Security & Permissions Configuration  To execute native BigQue...
  [0.3113] Environment Setup & Provisioning
           [Environment Setup & Provisioning] Environment Setup & Provisioning  This project is designed to run on the Google Cloud Platform (GCP), leveraging Go...
  [0.3612] Execution Order
           [Execution Order] Execution Order  To deploy and execute this pipeline end-to-end, run the scripts and workflows in the following order:  ⚙️ Environme...
  [0.3897] Execution Order
           [Execution Order] Upload all .sqlx transformation models into the definitions/ directory, maintaining the medallion subfolder hierarchy:  definitions/...
  [0.3994] Environment Setup & Provisioning > 3. Data Ingestion: Kaggle to BigQuery Pipeline > 

In [6]:
TEST_QUERIES = [
    "How is financial impact calculated from the model's predictions?"
]

for q in TEST_QUERIES:
    print("=" * 80)
    print("QUERY:", q)
    for row in vector_search(q, top_k=3):
        print(f"  [{row.distance:.4f}] {row.section}")
        print(f"           {row.chunk_text}")

QUERY: How is financial impact calculated from the model's predictions?
  [0.2459] Financial Impact Analysis
           [Financial Impact Analysis]
🛠️ Strategy (Test Data for Financial Projections): The financial evaluation is based exclusively on the test split (unseen holdout data) rather than the validation set. While hyperparameters and ensemble weights were optimized on the train/validation split, the validation set was still utilized for model selection. Calculating the financial impact on that same validation data could introduce selection bias and artificially inflate profit lift estimates. By evaluating the financial impact on the unseen test set, we ensure the most unbiased projection of actual production performance.

1. Key Assumptions

Lacking access to proprietary Instacart data, this model establishes baseline assumptions using publicly reported metrics. The financial projections are calculated by linking the Recall@5 performance of each model to three core business leve

## Pull every row for the spatial visualization

In [7]:
rows = list(client.query(f"""
    SELECT chunk_id, section, chunk_text, embedding
    FROM `{PROJECT}.vector_db.chunks_docs_embedded`
""").result())

print(f"{len(rows)} rows pulled")
print(f"embedding dimension: {len(rows[0].embedding)}")

66 rows pulled
embedding dimension: 3072


## Reduce to 2D with t-SNE

t-SNE is the standard tool for
visualizing whether embeddings cluster meaningfully. `perplexity` is tuned
down from sklearn's default (30, meant for hundreds+ of points) since this
corpus only has ~66 rows.

In [ ]:
import numpy as np
from sklearn.manifold import TSNE

embeddings = np.array([r.embedding for r in rows])

tsne = TSNE(n_components=2, perplexity=15, random_state=0, init="pca")
coords = tsne.fit_transform(embeddings)
print("coords shape:", coords.shape)

## Plot — color by top-level section, hover for details

Coloring by the top-level heading (the first segment of the breadcrumb) is
the actual sanity check: if retrieval quality is good, chunks under the same
top-level section should visually cluster together rather than scatter
randomly.

In [ ]:
import plotly.express as px

top_level = [(
    r.section.split(" > ")[0] if r.section else "(none)"
) for r in rows]
snippets = [r.chunk_text[:150].replace("\n", " ") + "..." for r in rows]

fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1],
    color=top_level,
    hover_name=[r.section for r in rows],
    hover_data={"snippet": snippets},
    title="chunks_docs_embedded -- t-SNE projection",
    labels={"x": "", "y": "", "color": "Top-level section"},
    width=1000, height=700,
)
fig.update_traces(marker=dict(size=10))
fig.show()